# 🎭 Deepfake Audio Detection System

## Google Colab Version — T4 GPU Optimized

This notebook is optimized for **T4 GPU** with direct dataset download from Kaggle.

### Setup Instructions:
1. Upload this notebook to Google Colab
2. Enable GPU: Runtime → Change runtime type → T4 GPU
3. Get Kaggle API key from kaggle.com → Account → API → Create New API Token
4. Upload kaggle.json when prompted
5. Run all cells

### Objectives:
- Classify audio as **Genuine (Human)** or **Deepfake (AI-Generated)**
- Achieve **≥80% Accuracy** and **≤12% EER**
- Build an end-to-end pipeline from feature extraction to deployment

### Dataset:
- **Fake-or-Real Dataset** from Kaggle (53,868 total samples)
- Using the `for-norm` (normalized) training directory
- **30,000 samples** used for training (balanced real/fake)

### Training Configuration:
- **Hardware:** T4 GPU (16GB VRAM)
- **Time Limit:** ~60 minutes
- **Batch Size:** 32
- **Max Epochs:** 40

## 1. Setup Google Colab Environment

In [ ]:
# Install required packages
!pip install -q librosa numpy pandas scikit-learn tensorflow matplotlib seaborn soundfile tqdm xgboost kaggle

In [ ]:
# Mount Google Drive (optional - for saving models)
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted!")

In [ ]:
# Setup Kaggle API using Colab secrets
import os
from pathlib import Path
from google.colab import userdata

print("Configuring Kaggle API using Colab secrets...")

# Get Kaggle credentials from Colab secrets
kaggle_username = userdata.get('KAGGLE_USER')
kaggle_key = userdata.get('KAGGLE_KEY')

# Create .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Write kaggle.json file
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

# Set permissions
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API configured successfully using secrets!")
print(f"Username: {kaggle_username}")

In [ ]:
# Download and extract dataset from Kaggle
import subprocess

print("Downloading Fake-or-Real dataset from Kaggle...")
print("This may take 2-5 minutes depending on your connection.")
print("="*60)

# Download dataset
!kaggle datasets download -d mohammedabdeldayem/the-fake-or-real-dataset

# Extract dataset
print("\nExtracting dataset...")
!unzip -q -o the-fake-or-real-dataset.zip -d /content/data

# List the extracted contents
print("\nDataset structure:")
!ls -la /content/data/
!ls -la /content/data/for-norm/ 2>/dev/null || echo "Checking alternative structure..."
!find /content/data -type d -maxdepth 3 | head -20

print("\n" + "="*60)
print("Dataset downloaded and extracted successfully!")

In [ ]:
# Check dataset structure and count files
import glob

# Find the correct path
data_paths = [
    "/content/data/for-norm/training",
    "/content/data/for-norm/for-norm/training",
    "/content/data/training",
]

data_dir = None
for path in data_paths:
    if os.path.exists(path):
        data_dir = path
        break

if data_dir is None:
    # Search for real and fake directories
    for root, dirs, files in os.walk('/content/data'):
        if 'real' in dirs and 'fake' in dirs:
            data_dir = root
            break

if data_dir:
    print(f"Dataset directory: {data_dir}")
    
    # Count files
    real_files = glob.glob(os.path.join(data_dir, 'real', '*.wav'))
    fake_files = glob.glob(os.path.join(data_dir, 'fake', '*.wav'))
    
    print(f"\nReal audio files: {len(real_files)}")
    print(f"Fake audio files: {len(fake_files)}")
    print(f"Total files: {len(real_files) + len(fake_files)}")
else:
    print("Could not find dataset directory. Please check the extraction.")
    print("\nContents of /content/data:")
    !ls -la /content/data/

## 2. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from scipy.optimize import brentq
from scipy.interpolate import interp1d

# XGBoost for comparison
import xgboost as xgb
import joblib

print(f"TensorFlow version: {tf.__version__}")
print(f"Librosa version: {librosa.__version__}")
print("All imports successful!")

In [ ]:
# ============================================
# HARDWARE INITIALIZATION (GPU/TPU)
# ============================================
import time

print("=" * 60)
print("INITIALIZING HARDWARE")
print("=" * 60)

# Check for GPU first
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU found: {gpus[0].name}")
    strategy = tf.distribute.get_strategy()
    IN_TPU = False
    IN_GPU = True
    
    # Enable memory growth for GPU
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth enabled for GPU")
else:
    # Try TPU
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        print(f"TPU found: {tpu.master()}")
        
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.experimental.TPUStrategy(tpu)
        
        print(f"Number of replicas: {strategy.num_replicas_in_sync}")
        IN_TPU = True
        IN_GPU = False
        
    except Exception as e:
        print(f"No GPU/TPU available: {e}")
        print("Using CPU...")
        strategy = tf.distribute.get_strategy()
        IN_TPU = False
        IN_GPU = False

print(f"Strategy: {strategy}")
print("=" * 60)

In [ ]:
# Enable mixed precision for TPU only (GPU already fast)
if IN_TPU:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print("Mixed precision enabled for TPU")
    print(f"Compute dtype: {policy.compute_dtype}")
    print(f"Variable dtype: {policy.variable_dtype}")
elif IN_GPU:
    print("GPU detected - using full precision for stability")
else:
    print("CPU mode - using full precision")

In [ ]:
# Configuration
class Config:
    # Paths (Google Colab)
    DATA_DIR = Path(data_dir)  # Set from previous cell
    REAL_DIR = DATA_DIR / "real"
    FAKE_DIR = DATA_DIR / "fake"
    MODELS_DIR = Path("/content/models")
    FIGURES_DIR = Path("/content/figures")
    
    # Audio parameters
    SAMPLE_RATE = 16000
    DURATION = 3  # seconds
    N_SAMPLES = SAMPLE_RATE * DURATION
    
    # Feature extraction
    N_MFCC = 40
    N_MELS = 128
    HOP_LENGTH = 512
    N_FFT = 2048
    
    # Model parameters (GPU-optimized)
    MAX_SAMPLES = 30000  # Use 30,000 samples for training
    BATCH_SIZE = 32  # T4 GPU optimized
    EPOCHS = 40  # Reduced to fit 1-hour time limit
    LEARNING_RATE = 0.001
    PATIENCE = 8  # Early stopping patience
    
    # Data split
    TEST_SIZE = 0.1
    VAL_SIZE = 0.1
    RANDOM_STATE = 42

config = Config()

# Create directories
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded successfully!")
print(f"Data directory: {config.DATA_DIR}")
print(f"Models directory: {config.MODELS_DIR}")
print(f"\nTraining Configuration:")
print(f"  MAX_SAMPLES: {config.MAX_SAMPLES}")
print(f"  BATCH_SIZE: {config.BATCH_SIZE}")
print(f"  EPOCHS: {config.EPOCHS}")
print(f"  PATIENCE: {config.PATIENCE}")
print(f"  Hardware: {'TPU' if IN_TPU else 'T4 GPU' if IN_GPU else 'CPU'}")

## 3. Data Loading and Exploration

In [ ]:
def load_dataset(data_dir):
    """Load dataset and create dataframe with file paths and labels."""
    real_dir = os.path.join(data_dir, 'real')
    fake_dir = os.path.join(data_dir, 'fake')
    
    data = []
    
    # Load real samples
    if os.path.exists(real_dir):
        for file in tqdm(glob.glob(os.path.join(real_dir, '*.wav')), desc="Loading real samples"):
            data.append({"path": file, "label": 1, "label_name": "real"})
    
    # Load fake samples
    if os.path.exists(fake_dir):
        for file in tqdm(glob.glob(os.path.join(fake_dir, '*.wav')), desc="Loading fake samples"):
            data.append({"path": file, "label": 0, "label_name": "fake"})
    
    df = pd.DataFrame(data)
    return df.sample(frac=1, random_state=config.RANDOM_STATE).reset_index(drop=True)

# Load dataset
df = load_dataset(config.DATA_DIR)
print(f"\nDataset loaded: {len(df)} samples")
print(f"\nClass distribution:")
print(df["label_name"].value_counts())

In [ ]:
# Display dataset info
print("Dataset Info:")
print(f"  Total samples: {len(df)}")
print(f"  Real samples: {len(df[df['label'] == 1])}")
print(f"  Fake samples: {len(df[df['label'] == 0])}")
print(f"\nUsing {config.MAX_SAMPLES} samples for training")
print(f"  Train: {int(config.MAX_SAMPLES * 0.8)} samples")
print(f"  Validation: {int(config.MAX_SAMPLES * 0.1)} samples")
print(f"  Test: {int(config.MAX_SAMPLES * 0.1)} samples")
print(f"\nSample file paths:")
df.head(10)

## 4. Audio Preprocessing

In [ ]:
def load_and_preprocess_audio(file_path, sr=config.SAMPLE_RATE, duration=config.DURATION):
    """Load and preprocess audio file."""
    try:
        # Load audio file
        audio, orig_sr = librosa.load(file_path, sr=sr, duration=duration)
        
        # Normalize amplitude
        audio = librosa.util.normalize(audio)
        
        # Trim silence
        audio, _ = librosa.effects.trim(audio, top_db=30)
        
        # Pad or truncate to fixed length
        if len(audio) > config.N_SAMPLES:
            audio = audio[:config.N_SAMPLES]
        else:
            audio = np.pad(audio, (0, config.N_SAMPLES - len(audio)), mode='constant')
        
        return audio
    except Exception as e:
        return None

print("Audio preprocessing function defined!")

In [ ]:
# Test audio loading with a sample
sample_path = df.iloc[0]['path']
sample_audio = load_and_preprocess_audio(sample_path)

if sample_audio is not None:
    print(f"Sample audio loaded successfully!")
    print(f"  Duration: {len(sample_audio)/config.SAMPLE_RATE:.2f} seconds")
    print(f"  Sample rate: {config.SAMPLE_RATE} Hz")
    print(f"  Samples: {len(sample_audio)}")
    
    # Plot waveform
    plt.figure(figsize=(12, 4))
    librosa.display.waveshow(sample_audio, sr=config.SAMPLE_RATE)
    plt.title('Sample Audio Waveform')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.tight_layout()
    plt.show()

## 5. Feature Extraction

In [ ]:
def extract_features(audio, sr=config.SAMPLE_RATE):
    """Extract comprehensive audio features."""
    features = {}
    
    # 1. MFCCs (Mel-Frequency Cepstral Coefficients)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=config.N_MFCC)
    features['mfccs_mean'] = np.mean(mfccs, axis=1)
    features['mfccs_std'] = np.std(mfccs, axis=1)
    
    # 2. Mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=config.N_MELS)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    features['mel_spec'] = mel_spec_db
    
    # 3. Spectral features
    features['spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
    features['spectral_bandwidth'] = np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr))
    features['spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr))
    features['spectral_contrast'] = np.mean(librosa.feature.spectral_contrast(y=audio, sr=sr))
    
    # 4. Zero crossing rate
    features['zcr'] = np.mean(librosa.feature.zero_crossing_rate(audio))
    
    # 5. RMS energy
    features['rms'] = np.mean(librosa.feature.rms(y=audio))
    
    # 6. Chroma features
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    features['chroma_mean'] = np.mean(chroma, axis=1)
    
    # 7. Tonnetz (tonal centroid features)
    tonnetz = librosa.feature.tonnetz(y=audio, sr=sr)
    features['tonnetz_mean'] = np.mean(tonnetz, axis=1)
    
    return features

print("Feature extraction function defined!")

In [ ]:
import gc
import pickle

FEATURES_CACHE_DIR = Path('/content/features_cache')
FEATURES_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def prepare_dataset(df, max_samples=None, batch_size=500):
    """Prepare dataset with extracted features using batch processing."""
    if max_samples is None:
        max_samples = len(df)
    
    cache_file = FEATURES_CACHE_DIR / f'features_{max_samples}.pkl'
    
    # Load cache if exists
    if cache_file.exists():
        print(f"Loading cached features from {cache_file}...")
        with open(cache_file, 'rb') as f:
            cached = pickle.load(f)
        print(f"Loaded {len(cached['labels'])} cached samples!")
        return cached['mfcc'], cached['mel'], cached['other'], cached['labels']
    
    mfcc_list = []
    mel_list = []
    other_list = []
    label_list = []
    
    df_subset = df.head(max_samples)
    total_batches = (len(df_subset) + batch_size - 1) // batch_size
    
    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min(start_idx + batch_size, len(df_subset))
        batch_df = df_subset.iloc[start_idx:end_idx]
        
        print(f"Processing batch {batch_num+1}/{total_batches} (samples {start_idx+1}-{end_idx})...")
        
        batch_mfcc = []
        batch_mel = []
        batch_other = []
        batch_labels = []
        
        for idx, row in batch_df.iterrows():
            audio = load_and_preprocess_audio(row['path'])
            if audio is not None:
                features = extract_features(audio)
                
                batch_mfcc.append(features['mfccs_mean'].astype(np.float32))
                batch_mel.append(features['mel_spec'].astype(np.float32))
                batch_other.append([
                    features['spectral_centroid'],
                    features['spectral_bandwidth'],
                    features['spectral_rolloff'],
                    features['spectral_contrast'],
                    features['zcr'],
                    features['rms']
                ])
                batch_labels.append(row['label'])
        
        mfcc_list.extend(batch_mfcc)
        mel_list.extend(batch_mel)
        other_list.extend(batch_other)
        label_list.extend(batch_labels)
        
        # Save progress after each batch
        progress_cache = FEATURES_CACHE_DIR / f'progress_{max_samples}.pkl'
        with open(progress_cache, 'wb') as f:
            pickle.dump({
                'mfcc': mfcc_list,
                'mel': mel_list,
                'other': other_list,
                'labels': label_list
            }, f)
        
        gc.collect()
        print(f"  Batch {batch_num+1} complete: {len(label_list)} samples processed")
    
    result = {
        'mfcc': np.array(mfcc_list, dtype=np.float32),
        'mel': np.array(mel_list, dtype=np.float32),
        'other': np.array(other_list, dtype=np.float32),
        'labels': np.array(label_list)
    }
    
    # Save final cache
    with open(cache_file, 'wb') as f:
        pickle.dump(result, f)
    print(f"\nFeatures cached to {cache_file}")
    
    # Clean up progress file
    progress_cache = FEATURES_CACHE_DIR / f'progress_{max_samples}.pkl'
    if progress_cache.exists():
        progress_cache.unlink()
    
    return result['mfcc'], result['mel'], result['other'], result['labels']

print("Dataset preparation function defined (optimized with batch processing)!")

In [ ]:
# Cache management
import shutil

cache_dir = Path('/content/features_cache')

# Check if cache exists
if cache_dir.exists():
    cache_files = list(cache_dir.glob('*.pkl'))
    total_size = sum(f.stat().st_size for f in cache_files) / 1e6
    print(f"Found {len(cache_files)} cache files ({total_size:.1f} MB)")
    
    # Option to clear cache
    clear_cache = False  # Set to True to clear cache and re-extract
    
    if clear_cache:
        shutil.rmtree(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        print("Cache cleared! Features will be re-extracted.")
    else:
        print("Using existing cache (set clear_cache=True to re-extract)")
else:
    cache_dir.mkdir(parents=True, exist_ok=True)
    print("No cache found. Features will be extracted fresh.")

In [ ]:
# Extract features (30,000 samples)
print(f"Extracting features from {config.MAX_SAMPLES} samples...")
print("Features are cached - will be fast on re-run!")
print("="*60)

feature_start_time = time.time()

mfcc_features, mel_features, other_features, labels = prepare_dataset(
    df, max_samples=config.MAX_SAMPLES, batch_size=500
)

feature_time = time.time() - feature_start_time
print(f"\n{'='*60}")
print(f"Feature extraction complete in {feature_time/60:.1f} minutes!")
print(f"{'='*60}")
print(f"\nFeature shapes:")
print(f"  MFCC features: {mfcc_features.shape} ({mfcc_features.nbytes/1e6:.1f} MB)")
print(f"  Mel-spectrogram: {mel_features.shape} ({mel_features.nbytes/1e6:.1f} MB)")
print(f"  Other features: {other_features.shape} ({other_features.nbytes/1e6:.1f} MB)")
print(f"  Labels: {labels.shape}")
print(f"\nLabel distribution:")
print(f"  Real: {np.sum(labels==1)} samples")
print(f"  Fake: {np.sum(labels==0)} samples")
print(f"\nTotal memory used: {(mfcc_features.nbytes + mel_features.nbytes + other_features.nbytes)/1e6:.1f} MB")

## 6. Data Visualization

In [ ]:
def plot_feature_comparison(mfcc_features, labels):
    """Plot feature comparison between real and fake audio."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    real_features = mfcc_features[labels == 1]
    fake_features = mfcc_features[labels == 0]
    
    # Plot first 6 MFCC coefficients
    for i in range(6):
        ax = axes[i // 3, i % 3]
        ax.hist(real_features[:, i], bins=30, alpha=0.5, label='Real', density=True)
        ax.hist(fake_features[:, i], bins=30, alpha=0.5, label='Fake', density=True)
        ax.set_title(f'MFCC Coefficient {i+1}')
        ax.legend()
    
    plt.suptitle('MFCC Feature Distribution: Real vs Fake', fontsize=14)
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / 'mfcc_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_feature_comparison(mfcc_features, labels)

In [ ]:
def plot_mel_spectrogram_comparison(mel_features, labels):
    """Plot mel-spectrogram comparison."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Real sample
    real_idx = np.where(labels == 1)[0][0]
    librosa.display.specshow(mel_features[real_idx], sr=config.SAMPLE_RATE,
                            x_axis='time', y_axis='mel', ax=axes[0])
    axes[0].set_title('Real Audio - Mel Spectrogram')
    axes[0].set_xlabel('Time')
    axes[0].set_ylabel('Mel Frequency')
    
    # Fake sample
    fake_idx = np.where(labels == 0)[0][0]
    librosa.display.specshow(mel_features[fake_idx], sr=config.SAMPLE_RATE,
                            x_axis='time', y_axis='mel', ax=axes[1])
    axes[1].set_title('Fake Audio - Mel Spectrogram')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Mel Frequency')
    
    plt.suptitle('Mel-Spectrogram Comparison', fontsize=14)
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / 'mel_spectrogram_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_mel_spectrogram_comparison(mel_features, labels)

## 7. Data Preparation for Training

In [ ]:
# Prepare data for training
import gc

print("Preparing data for training...")
print("="*60)

# Combine features for classical ML (use float32 to save memory)
X_classical = np.hstack([mfcc_features, other_features]).astype(np.float32)
y = labels

# Reshape mel features for CNN+LSTM (use float32)
X_cnn = mel_features[..., np.newaxis].astype(np.float32)

# Free memory
del mfcc_features, mel_features, other_features
gc.collect()

# Split data: 80% train, 10% val, 10% test
X_train_cnn, X_test_cnn, y_train, y_test = train_test_split(
    X_cnn, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)

X_train_cnn, X_val_cnn, y_train, y_val = train_test_split(
    X_train_cnn, y_train, test_size=config.VAL_SIZE/(1-config.TEST_SIZE),
    random_state=config.RANDOM_STATE, stratify=y_train
)

# Split classical features
X_train_class, X_test_class, _, _ = train_test_split(
    X_classical, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)
X_train_class, X_val_class, _, _ = train_test_split(
    X_train_class, y_train, test_size=config.VAL_SIZE/(1-config.TEST_SIZE),
    random_state=config.RANDOM_STATE, stratify=y_train
)

# Free original data
del X_cnn, X_classical, y
gc.collect()

# Scale classical features
scaler = StandardScaler()
X_train_class_scaled = scaler.fit_transform(X_train_class).astype(np.float32)
X_val_class_scaled = scaler.transform(X_val_class).astype(np.float32)
X_test_class_scaled = scaler.transform(X_test_class).astype(np.float32)

# Free unscaled data
del X_train_class, X_val_class, X_test_class
gc.collect()

print("Data split complete!")
print(f"\nCNN+LSTM Data:")
print(f"  Train: {X_train_cnn.shape} ({X_train_cnn.nbytes/1e6:.1f} MB)")
print(f"  Val: {X_val_cnn.shape} ({X_val_cnn.nbytes/1e6:.1f} MB)")
print(f"  Test: {X_test_cnn.shape} ({X_test_cnn.nbytes/1e6:.1f} MB)")
print(f"\nClassical ML Data:")
print(f"  Train: {X_train_class_scaled.shape} ({X_train_class_scaled.nbytes/1e6:.1f} MB)")
print(f"  Val: {X_val_class_scaled.shape} ({X_val_class_scaled.nbytes/1e6:.1f} MB)")
print(f"  Test: {X_test_class_scaled.shape} ({X_test_class_scaled.nbytes/1e6:.1f} MB)")
print(f"\nTotal memory: {(X_train_cnn.nbytes + X_val_cnn.nbytes + X_test_cnn.nbytes + X_train_class_scaled.nbytes + X_val_class_scaled.nbytes + X_test_class_scaled.nbytes)/1e6:.1f} MB")

## 8. Deep Learning Model (CNN + LSTM) — TPU Optimized

In [ ]:
# Build model within TPU strategy scope
with strategy.scope():
    def build_cnn_lstm_model(input_shape):
        """Build CNN + LSTM model for deepfake detection."""
        model = models.Sequential([
            # CNN Feature Extractor
            layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            layers.Conv2D(128, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            
            # Reshape for LSTM
            layers.Reshape((-1, 128)),
            
            # Temporal Learning
            layers.LSTM(128, return_sequences=False),
            layers.Dropout(0.5),
            
            # Classifier
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        
        return model

    # Build model
    input_shape = (128, 128, 1)  # Mel-spectrogram shape
    model = build_cnn_lstm_model(input_shape)
    model.summary()

In [ ]:
# Train CNN+LSTM model
print("Training CNN+LSTM model...")
print("="*60)
print(f"Dataset size: {config.MAX_SAMPLES} samples")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Max epochs: {config.EPOCHS}")
print(f"Time limit: 55 minutes")
print("="*60)

# Time monitoring callback
class TimeMonitor(callbacks.Callback):
    def __init__(self, max_time_minutes=55):
        self.max_time_seconds = max_time_minutes * 60
        self.start_time = None
    
    def on_train_begin(self, logs=None):
        self.start_time = time.time()
    
    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self.start_time
        remaining = self.max_time_seconds - elapsed
        
        if remaining < 120:  # Less than 2 minutes left
            print(f"\nTime limit approaching ({remaining/60:.1f} min remaining)")
            self.model.stop_training = True

# Callbacks
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=config.PATIENCE,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

model_checkpoint = callbacks.ModelCheckpoint(
    config.MODELS_DIR / 'deepfake_cnn_lstm_best.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

time_monitor = TimeMonitor(max_time_minutes=55)

# Train
training_start_time = time.time()

history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    callbacks=[early_stop, reduce_lr, model_checkpoint, time_monitor],
    verbose=1
)

training_time = time.time() - training_start_time
print(f"\nTraining completed in {training_time/60:.1f} minutes")

# Save final model
model.save(config.MODELS_DIR / 'deepfake_cnn_lstm_final.h5')
print("Model saved successfully!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Classical ML Model (XGBoost)

In [ ]:
# Train XGBoost model
print("Training XGBoost model...")
print("="*60)

# Create and train XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=config.RANDOM_STATE
)

xgb_start_time = time.time()

xgb_model.fit(
    X_train_class_scaled, y_train,
    eval_set=[(X_val_class_scaled, y_val)],
    verbose=50
)

xgb_time = time.time() - xgb_start_time
print(f"\nXGBoost training completed in {xgb_time:.1f} seconds")

# Save model
joblib.dump(xgb_model, config.MODELS_DIR / 'deepfake_xgboost.pkl')
joblib.dump(scaler, config.MODELS_DIR / 'scaler.pkl')
print("XGBoost model saved successfully!")

## 10. Model Evaluation

In [ ]:
def calculate_eer(y_true, y_score):
    """Calculate Equal Error Rate (EER)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return eer

def evaluate_model(y_true, y_pred, y_prob, model_name):
    """Comprehensive model evaluation."""
    results = {}
    
    # Overall metrics
    results['accuracy'] = accuracy_score(y_true, y_pred)
    results['f1'] = f1_score(y_true, y_pred)
    results['precision'] = precision_score(y_true, y_pred)
    results['recall'] = recall_score(y_true, y_pred)
    
    # EER
    results['eer'] = calculate_eer(y_true, y_prob)
    
    # Per-class accuracy
    cm = confusion_matrix(y_true, y_pred)
    results['per_class_accuracy'] = cm.diagonal() / cm.sum(axis=1)
    
    # ROC-AUC
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    results['roc_auc'] = auc(fpr, tpr)
    results['fpr'] = fpr
    results['tpr'] = tpr
    
    # Print results
    print(f"\n{'='*60}")
    print(f"{model_name} Results")
    print(f"{'='*60}")
    print(f"Overall Accuracy: {results['accuracy']*100:.2f}%")
    print(f"F1 Score: {results['f1']*100:.2f}%")
    print(f"Precision: {results['precision']*100:.2f}%")
    print(f"Recall: {results['recall']*100:.2f}%")
    print(f"EER: {results['eer']*100:.2f}%")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")
    print(f"\nPer-Class Accuracy:")
    print(f"  Fake: {results['per_class_accuracy'][0]*100:.2f}%")
    print(f"  Real: {results['per_class_accuracy'][1]*100:.2f}%")
    print(f"\nConfusion Matrix:")
    print(cm)
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Fake', 'Real']))
    
    return results

print("Evaluation functions defined!")

In [ ]:
# Evaluate CNN+LSTM model
y_prob_cnn = model.predict(X_test_cnn, verbose=0).flatten()
y_pred_cnn = (y_prob_cnn > 0.5).astype(int)

# Evaluate
cnn_results = evaluate_model(y_test, y_pred_cnn, y_prob_cnn, "CNN+LSTM")

In [ ]:
# Evaluate XGBoost model
y_prob_xgb = xgb_model.predict_proba(X_test_class_scaled)[:, 1]
y_pred_xgb = xgb_model.predict(X_test_class_scaled)

# Evaluate
xgb_results = evaluate_model(y_test, y_pred_xgb, y_prob_xgb, "XGBoost")

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CNN+LSTM confusion matrix
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
axes[0].set_title('CNN+LSTM Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# XGBoost confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
axes[1].set_title('XGBoost Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot ROC curves
plt.figure(figsize=(8, 6))

plt.plot(cnn_results['fpr'], cnn_results['tpr'],
         label=f"CNN+LSTM (AUC = {cnn_results['roc_auc']:.3f})")

plt.plot(xgb_results['fpr'], xgb_results['tpr'],
         label=f"XGBoost (AUC = {xgb_results['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(True)
plt.savefig(config.FIGURES_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Model Comparison and Requirements Check

In [ ]:
# Compare models
comparison_data = [
    {
        'Model': 'CNN+LSTM',
        'Accuracy': cnn_results['accuracy'],
        'F1 Score': cnn_results['f1'],
        'EER': cnn_results['eer'],
        'ROC-AUC': cnn_results['roc_auc'],
        'Fake Accuracy': cnn_results['per_class_accuracy'][0],
        'Real Accuracy': cnn_results['per_class_accuracy'][1]
    },
    {
        'Model': 'XGBoost',
        'Accuracy': xgb_results['accuracy'],
        'F1 Score': xgb_results['f1'],
        'EER': xgb_results['eer'],
        'ROC-AUC': xgb_results['roc_auc'],
        'Fake Accuracy': xgb_results['per_class_accuracy'][0],
        'Real Accuracy': xgb_results['per_class_accuracy'][1]
    }
]

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Model')

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string())

# Check if requirements are met
print("\n" + "="*60)
print("REQUIREMENTS CHECK")
print("="*60)

best_model = comparison_df['Accuracy'].idxmax()
best_results = comparison_df.loc[best_model]

checks = [
    ('Overall Accuracy >= 80%', best_results['Accuracy'] >= 0.80),
    ('EER <= 12%', best_results['EER'] <= 0.12),
    ('F1 Score >= 80%', best_results['F1 Score'] >= 0.80),
    ('Fake Accuracy >= 75%', best_results['Fake Accuracy'] >= 0.75),
    ('Real Accuracy >= 75%', best_results['Real Accuracy'] >= 0.75)
]

all_passed = True
for check_name, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  {check_name}: {status}")
    if not passed:
        all_passed = False

print(f"\nBest Model: {best_model}")
print(f"\n{'='*60}")
if all_passed:
    print("ALL REQUIREMENTS MET!")
else:
    print("Some requirements not met. Consider more training data or tuning.")
print("="*60)

## 12. Save and Export Models

In [ ]:
# Save all artifacts
print("Saving model artifacts...")

# Save label encoder
le = LabelEncoder()
le.fit(['fake', 'real'])
joblib.dump(le, config.MODELS_DIR / 'label_encoder.pkl')

# Save config
config_dict = {
    'sample_rate': config.SAMPLE_RATE,
    'duration': config.DURATION,
    'n_mfcc': config.N_MFCC,
    'n_mels': config.N_MELS,
    'hop_length': config.HOP_LENGTH,
    'n_fft': config.N_FFT
}
joblib.dump(config_dict, config.MODELS_DIR / 'config.pkl')

print(f"\nAll artifacts saved to: {config.MODELS_DIR}")
print("Files saved:")
for file in os.listdir(config.MODELS_DIR):
    print(f"  - {file}")

In [ ]:
# Copy models to Google Drive for persistence
import shutil

drive_models_dir = "/content/drive/MyDrive/deepfake_models"
os.makedirs(drive_models_dir, exist_ok=True)

# Copy all model files
for file in os.listdir(config.MODELS_DIR):
    src = os.path.join(config.MODELS_DIR, file)
    dst = os.path.join(drive_models_dir, file)
    shutil.copy2(src, dst)

print(f"Models copied to Google Drive: {drive_models_dir}")
print("\nFiles in Google Drive:")
for file in os.listdir(drive_models_dir):
    print(f"  - {file}")

## 13. Inference Function

In [ ]:
def predict_audio(file_path, model, scaler=None, config_dict=None):
    """Predict if audio is real or fake."""
    # Load and preprocess audio
    audio = load_and_preprocess_audio(file_path)
    if audio is None:
        return None
    
    # Extract features
    features = extract_features(audio)
    
    # For CNN+LSTM model
    if hasattr(model, 'predict') and len(model.input_shape) == 5:  # CNN
        mel_spec = features['mel_spec']
        mel_spec = mel_spec[..., np.newaxis]
        mel_spec = np.expand_dims(mel_spec, axis=0)
        prob = model.predict(mel_spec, verbose=0)[0][0]
    else:  # Classical ML
        mfcc_features = features['mfccs_mean']
        other_features_arr = [
            features['spectral_centroid'],
            features['spectral_bandwidth'],
            features['spectral_rolloff'],
            features['spectral_contrast'],
            features['zcr'],
            features['rms']
        ]
        X = np.hstack([mfcc_features, other_features_arr]).reshape(1, -1)
        if scaler:
            X = scaler.transform(X)
        prob = model.predict_proba(X)[0][1]
    
    # Determine prediction
    prediction = 'real' if prob > 0.5 else 'fake'
    confidence = prob if prob > 0.5 else 1 - prob
    
    return {
        'prediction': prediction,
        'confidence': float(confidence),
        'probability_real': float(prob),
        'probability_fake': float(1 - prob)
    }

print("Inference function defined!")

In [ ]:
# Test inference on sample files
print("Testing inference...")
print("="*60)

# Test with a real sample
real_sample = df[df['label'] == 1].iloc[0]['path']
result = predict_audio(real_sample, model)
print(f"\nReal sample:")
print(f"  Path: {real_sample}")
print(f"  Prediction: {result['prediction'].upper()}")
print(f"  Confidence: {result['confidence']*100:.2f}%")

# Test with a fake sample
fake_sample = df[df['label'] == 0].iloc[0]['path']
result = predict_audio(fake_sample, model)
print(f"\nFake sample:")
print(f"  Path: {fake_sample}")
print(f"  Prediction: {result['prediction'].upper()}")
print(f"  Confidence: {result['confidence']*100:.2f}%")

## 14. Download Models for Local Use

In [ ]:
# Create a zip file of all models
import zipfile

zip_path = '/content/deepfake_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(config.MODELS_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, config.MODELS_DIR)
            zipf.write(file_path, arcname)

print(f"Models zipped: {zip_path}")

# Download the zip file
from google.colab import files
files.download(zip_path)

print("\nDownload started! Check your downloads folder.")

## 15. Summary

In [ ]:
total_time = (time.time() - feature_start_time) / 60

print("\n" + "="*60)
print("DEEPFAKE AUDIO DETECTION - TRAINING COMPLETE")
print("="*60)
print(f"\nTotal time: {total_time:.1f} minutes")
print(f"Dataset used: {config.MAX_SAMPLES} samples")
print(f"Hardware: {'TPU v5e-1' if IN_TPU else 'T4 GPU' if IN_GPU else 'CPU'}")
print(f"\nModels trained and saved:")
print(f"  1. CNN+LSTM: {config.MODELS_DIR / 'deepfake_cnn_lstm_final.h5'}")
print(f"  2. XGBoost: {config.MODELS_DIR / 'deepfake_xgboost.pkl'}")
print(f"\nArtifacts saved:")
print(f"  - Scaler: {config.MODELS_DIR / 'scaler.pkl'}")
print(f"  - Label Encoder: {config.MODELS_DIR / 'label_encoder.pkl'}")
print(f"  - Config: {config.MODELS_DIR / 'config.pkl'}")
print(f"\nModels also saved to Google Drive: {drive_models_dir}")
print(f"\nFigures saved to: {config.FIGURES_DIR}")
print("\nNext steps:")
print("  1. Download the models zip file (cell above)")
print("  2. Extract to 'models/' folder in your local project")
print("  3. Run test_audio.py to test with new audio files")
print("  4. Deploy app.py using Streamlit")